## Analyze Predictive Words

In [1]:
import pandas as pd

import sys
import os

sys.path.append(os.path.abspath("../src"))

In [4]:
from sentiment_model import SentimentModel
import torch.nn as nn
import torch
from tokenizer import Tokenizer



In [28]:
model = SentimentModel(vocab_size=10000,embedding_dim=100)
    
model.load_state_dict(torch.load("./../checkpoints/sentiment_model.pth",map_location='cpu'))
model.eval()

SentimentModel(
  (embedding): Embedding(10000, 100)
  (fc1): Linear(in_features=100, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=2, bias=True)
  (Relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
)

In [29]:
tokenizer = Tokenizer()

tokenizer.load()



In [30]:
word_influences = []

with torch.no_grad():
    for token_id,word in tokenizer.id_to_word.items():
       #1. Turn the ID into a tensor [[token_id]]
       input_tensor = torch.tensor([[token_id]]).long()
       
       #2. Get the model's output
       logits = model(input_tensor)
       
       #3. GEt teh probability of "Positive" (Index 1)
       probs = torch.softmax(logits, dim=1).squeeze()       
       #print(probs)
       pos_score =probs[1].item()
       
       
       word_influences.append((word,pos_score))
       
        
        
        

In [31]:
#sort in descending order (Highest to lowest)
word_influences.sort(key=lambda x:x[1],reverse=True)

print(word_influences[:10])
print(word_influences[-20:])

[('and', 1.0), ('my', 1.0), ('well', 1.0), ('great', 1.0), ('think', 1.0), ('seen', 1.0), ('love', 1.0), ('best', 1.0), ('ever', 1.0), ('still', 1.0)]
[('mst3k', 1.7188841239944613e-38), ('210', 1.1145636515159015e-38), ('poor', 1.2004783814024275e-40), ('lousy', 1.106325137584443e-40), ('laughable', 1.7191129560336856e-41), ('poorly', 1.0986179960306566e-41), ('horrible', 1.6535321879032841e-43), ('forgettable', 1.6815581571897805e-44), ('disappointing', 9.80908925027372e-45), ('310', 8.407790785948902e-45), ('dull', 2.802596928649634e-45), ('disappointment', 1.401298464324817e-45), ('worst', 0.0), ('boring', 0.0), ('terrible', 0.0), ('awful', 0.0), ('waste', 0.0), ('lame', 0.0), ('fails', 0.0), ('410', 0.0)]


In [33]:
#save words

def save_predictive_words():
    with open("./../experiments/predictive_words.txt","w",encoding="utf-8") as f:
        f.write(f"Positive words: \n")
        f.write(" \n")
        for word,prob in word_influences[:10]:
            f.write(f"{word}\n")
        
        f.write(" \n")
        f.write(" \n")

        f.write(f"Negative words: \n")
        f.write(" \n")
        for word,prob in word_influences[-10:]:
            f.write(f"{word}\n")

In [34]:
save_predictive_words()

## PREDICTING

In [36]:
import html

def visualize_attention(sentence, model, tokenizer, filename="experiments/attention_examples.html"):
    model.eval()
    words = sentence.split()
    highlights = []

    with torch.no_grad():
        for word in words:
            # Get the positive score for this specific word
            _, score = model.predict(word, tokenizer)
            # We normalize the score for the background-color (0.5 is neutral)
            # Higher than 0.5 = Green (Positive), Lower than 0.5 = Red (Negative)
            highlights.append((word, score))

    # Generate HTML string
    html_output = "<html><body style='font-family: sans-serif; padding: 20px;'>"
    html_output += f"<h3>Sentence: \"{sentence}\"</h3><div style='line-height: 2.5;'>"

    for word, score in highlights:
        # Map score to color intensity
        if score > 0.5:
            # Green highlight for positive influence
            alpha = (score - 0.5) * 2  # Scale 0.5-1.0 to 0.0-1.0
            color = f"rgba(0, 255, 0, {alpha:.2f})"
        else:
            # Red highlight for negative influence
            alpha = (0.5 - score) * 2  # Scale 0.0-0.5 to 0.0-1.0
            color = f"rgba(255, 0, 0, {alpha:.2f})"
            
        html_output += f"<span style='background-color: {color}; padding: 5px; margin: 2px; border-radius: 3px;'>{html.escape(word)}</span> "

    html_output += "</div></body></html>"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(html_output)
    print(f"✅ Visualization saved to {filename}")

In [ ]:
visualize_attention("This movie was absolutely amazing",model,tokenizer,"./../experiments/attention_examples.html")